# Label flipping: o'qitish ma'lumotini zaharlash

Model o'zi o'rgangan ma'lumot qanchalik to'g'ri bo'lsa, shunchalik to'g'ri ishlaydi. Tasavvur qiling: siz shu
ma'lumotning kichik bir qismiga, bor-yo'g'i olti foiziga, yozish huquqini qo'lga kiritdingiz. Butun modelni
buzmoqchi emassiz, nozikroq ish qilmoqchisiz: model bitta aniq nuqta bo'yicha xato javob bersin, qolgan hamma
joyda esa avvalgidek ishlayversin. Shunda hech kim hech narsani sezmaydi.

Bunday hujum **data poisoning** (ma'lumotni zaharlash) deyiladi. Uning eng oddiy turi **label flipping**:
ma'lumotning o'ziga tegmaysiz, faqat bir nechta javobni (label'ni) teskarisiga almashtirasiz.

Bu darsda uch narsani o'rganasiz:

1. Model yangi nuqta haqida uning «qo'shnilari»ga qarab xulosa chiqarishini.
2. Nega tavakkal almashtirilgan label'lar hech narsa bermasligini.
3. Nega nishonli zaharlash umumiy aniqlikka deyarli ta'sir qilmasligini va shuning uchun xavfli ekanini.

Python'da ro'yxat, `for` sikli va ro'yxatni tartiblashni bilsangiz yetarli.

Bir gap: bu o'quv muhiti. Birovning ma'lumotini ruxsatsiz o'zgartirish jinoyat hisoblanadi.

## 1. Labga ulanamiz

Platformadagi lab sahifasini oching. U yerda **Token** degan maydon bor: undagi satrni nusxalab, pastdagi
`TOKEN` ichiga qo'ying. Token faqat sizga tegishli va bir necha soatdan keyin eskiradi. Eskirib qolsa, lab
sahifasini yangilab, yangisini oling.

In [ ]:
BASE = "https://platform.hacknow.uz/ai-poligon"
TOKEN = ""   # lab sahifasidagi Token maydonidan nusxalab, qo'shtirnoq ichiga qo'ying

assert TOKEN, "Avval TOKEN ni to'ldiring: platformadagi lab sahifasidan nusxalang."

import math
import urllib.request

urllib.request.urlretrieve(BASE + "/client/hn_lab.py", "hn_lab.py")   # labning kichik mijozi
from hn_lab import HnLab

lab = HnLab(BASE, "ml-label-flip", TOKEN)
info = lab.task()
print(info["lab"]["title"])
print("Urinish:", info["attempt"], "| sarflangan so'rov:", info["queries_used"], "/", info["query_budget"])
if info["solved"]:
    print("Bu lab sizda allaqachon yechilgan. Flag:", info.get("flag"))
if info["query_budget"] - info["queries_used"] < 5:
    print("Diqqat: so'rovlar kam qoldi. Yetmay qolsa, lab.reset() bilan yangi urinish boshlang va notebook'ni boshidan yurgizing.")

## 2. Vazifa bilan tanishamiz

O'qitish to'plamida 200 ta nuqta bor. Har bir nuqta ikkita sondan (`x1`, `x2`) va bitta javobdan iborat:
label 0 yoki 1. Model shu nuqtalarga qarab ikki guruh orasidan chegara o'tkazadi va yangi nuqta chegaraning
qaysi tomoniga tushsa, o'sha guruhga tegishli deb aytadi.

Sizga bitta **nishon** nuqta berilgan. Toza model uni to'g'ri taniydi. Siz ko'pi bilan 12 ta label'ni
almashtira olasiz. Shundan keyin model qayta o'qitiladi va ikki shart tekshiriladi: nishon endi noto'g'ri
tasniflanishi kerak, yashirin test to'plamidagi aniqlik esa sezilarli tushmasligi kerak.

In [ ]:
vazifa = info["task"]
nishon = vazifa["target"]["x"]
nishon_label = vazifa["target"]["label"]

print("O'qitish to'plami:", vazifa["train_size"], "ta nuqta | almashtirish mumkin:", vazifa["max_flips"], "ta label")
print("Nishon nuqta:", nishon, "| uning to'g'ri label'i:", nishon_label)
print("Toza modelning aniqligi:", vazifa["base_clean_accuracy"], "| ruxsat etilgan eng past aniqlik:", vazifa["min_clean_accuracy"])

So'rovlar cheklangan: bitta urinishda 40 marta javob yubora olasiz. So'rovlar tugab qolsa, `lab.reset()` yangi
urinish boshlaydi, lekin unda nishon ham yangilanadi.

## 3. Ma'lumotni yuklab olamiz

O'qitish to'plami ochiq: uni `train.csv` fayli sifatida olish mumkin. Faylni yuklab olish so'rov sarflamaydi.

In [ ]:
matn = lab.artifact("train.csv").decode()

nuqtalar = []   # har element: (indeks, x1, x2, label)
for satr in matn.strip().splitlines()[1:]:
    idx, x1, x2, label = satr.split(",")
    nuqtalar.append((int(idx), float(x1), float(x2), int(label)))

print(len(nuqtalar), "ta nuqta yuklandi. Birinchi uchtasi:")
for nuqta in nuqtalar[:3]:
    print("  ", nuqta)
print("label 0:", sum(1 for nuqta in nuqtalar if nuqta[3] == 0), "ta | label 1:", sum(1 for nuqta in nuqtalar if nuqta[3] == 1), "ta")

Rasm chizib ko'rsak, hammasi ancha tushunarli bo'ladi. Keyingi katak ikki guruhni va nishonni chizadi. Agar
kompyuteringizda `matplotlib` o'rnatilmagan bo'lsa, katak shunchaki o'tkazib yuboriladi: dars rasmsiz ham davom
etadi.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib o'rnatilmagan: rasm chizilmaydi, lekin dars davom etadi.")

if plt:
    plt.figure(figsize=(6, 5))
    for qiymat, rang in ((0, "tab:blue"), (1, "tab:orange")):
        guruh = [nuqta for nuqta in nuqtalar if nuqta[3] == qiymat]
        plt.scatter([nuqta[1] for nuqta in guruh], [nuqta[2] for nuqta in guruh], s=14, color=rang, label="label %d" % qiymat)
    plt.scatter([nishon[0]], [nishon[1]], s=220, marker="*", color="red", edgecolors="black", label="nishon")
    plt.legend()
    plt.title("O'qitish to'plami va nishon")
    plt.show()

## 4. Masofani o'lchaymiz

Bizga ikki nuqta bir-biridan qanchalik uzoqligini o'lchaydigan funksiya kerak bo'ladi. Bu maktabdagi Pifagor
teoremasining o'zi. Sinash uchun nishondan to'plamdagi eng birinchi nuqtagacha bo'lgan masofani hisoblaymiz.

In [ ]:
def masofa(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


idx, x1, x2, label = nuqtalar[0]
print("0-nuqta:", (x1, x2), "| label:", label, "| nishongacha masofa:", round(masofa((x1, x2), nishon), 3))

## 5. Sinov so'rovi

Javob qanday ko'rinishda kelishini bilish uchun bitta sinov yuboramiz. Ataylab foydasiz narsa tanlaymiz:
nishondan eng uzoqda turgan bitta nuqtaning label'ini almashtiramiz.

In [ ]:
eng_uzoq = max(nuqtalar, key=lambda nuqta: masofa((nuqta[1], nuqta[2]), nishon))[0]

javob = lab.score({"flip": [eng_uzoq]})
print(javob["output"])
print(javob["message"])
print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

Javobda `target_pred` model nishonni qaysi guruhga qo'shganini, `target_true` esa to'g'ri javobni ko'rsatadi.
Ular hozir bir xil: nishon hali to'g'ri tasniflanyapti. `clean_accuracy` zaharlangan modelning yashirin testdagi
aniqligi, u `min_clean_accuracy` dan past tushmasligi kerak.

Uzoqdagi bitta label hech narsani o'zgartirmadi. Tavakkal tanlangan o'n ikkita label ham deyarli hech narsa
bermaydi: ular to'plam bo'ylab sochilib ketadi va chegara joyidan qimirlamaydi.

**Unda nima qilish kerak?** Model nishon haqida uning atrofidagi nuqtalarga qarab xulosa chiqaradi. Nishonning
yonidagi, u bilan bir guruhdagi nuqtalar «bu yer bizning hudud» deb turibdi. Aynan o'shalarning label'ini
teskarisiga almashtirsak, model nishon atrofini boshqa guruhning hududi deb o'rganadi va chegarani nishonning
narigi tomoniga suradi. To'plamning qolgan qismi o'zgarmagani uchun umumiy aniqlik deyarli tushmaydi.

## 6. Topshiriq 1: nishonning eng yaqin qo'shnilarini toping

Nishon bilan **bir xil label'li** hamma nuqtalar uchun nishongacha bo'lgan masofani hisoblang va ularni
yaqinidan uzog'iga qarab tartiblang.

Pastdagi katakda `yaqinlar` ro'yxatini to'ldiring. Har element juftlik bo'lsin: `(masofa, indeks)`. Juftlikda
masofa birinchi turgani muhim: Python juftliklar ro'yxatini birinchi elementiga qarab tartiblaydi, ya'ni
`yaqinlar.sort()` deb yozsangiz, eng yaqin nuqta ro'yxat boshiga chiqadi.

Yordam: nuqtaning label'i nishonniki bilan bir xilligini `label == nishon_label` deb tekshirasiz.

In [ ]:
yaqinlar = []   # har element: (nishongacha masofa, nuqta indeksi) — faqat nishon bilan bir xil label'li nuqtalar

# TOPSHIRIQ 1: siklni va tartiblashni shu yerga yozing.

Keyingi katak ro'yxatingizni tekshiradi. Bu tekshiruv so'rov sarflamaydi.

In [ ]:
def yaqinlar_togri(royxat):
    bir_xil = [nuqta[0] for nuqta in nuqtalar if nuqta[3] == nishon_label]
    if len(royxat) != len(bir_xil):
        print("Hali tayyor emas: ro'yxatda", len(royxat), "ta element bor, kerak:", len(bir_xil),
              "(nishon bilan bir xil label'li nuqtalar soni).")
        return False
    if sorted(indeks for _, indeks in royxat) != bir_xil:
        print("Xato: ro'yxatga faqat nishon bilan bir xil label'li nuqtalar kirishi kerak, har biri bir martadan.")
        return False
    if [m for m, _ in royxat] != sorted(m for m, _ in royxat):
        print("Xato: ro'yxat masofa bo'yicha tartiblanmagan. Oxirida yaqinlar.sort() ni chaqirdingizmi?")
        return False
    return True


if yaqinlar_togri(yaqinlar):
    print("Ro'yxat to'g'ri. Eng yaqin beshta qo'shni (masofa, indeks):")
    for m, indeks in yaqinlar[:5]:
        print("   %.3f  %d" % (m, indeks))

## 7. Topshiriq 2: label'larni tanlang va yuboring

Endi `tanlov` ro'yxatiga almashtirmoqchi bo'lgan nuqtalaringizning indekslarini yozing. Eng yaqin o'n ikkita
qo'shnidan boshlang: `yaqinlar` ning boshidagi o'n ikkita juftlikdan indekslarni oling.

In [ ]:
tanlov = []   # almashtiriladigan nuqtalarning indekslari (ko'pi bilan 12 ta)

# TOPSHIRIQ 2: tanlovni shu yerga yozing.

In [ ]:
if not tanlov:
    print("Hali hech narsa tanlamadingiz: Topshiriq 2 ni bajaring.")
elif len(tanlov) > vazifa["max_flips"] or len(set(tanlov)) != len(tanlov):
    print("Xato: ko'pi bilan", vazifa["max_flips"], "ta indeks bo'lishi va ular takrorlanmasligi kerak.")
else:
    javob = lab.score({"flip": [int(indeks) for indeks in tanlov]})
    print(javob["output"])
    print(javob["message"])
    if javob["solved"]:
        print("Bo'ldi! Model nishonni noto'g'ri tasnifladi, aniqlik esa chegarada qoldi.")
        print("Flag:", javob["flag"])

Flag chiqqan bo'lsa, uni platformadagi topshiriq sahifasiga yuboring. Flag faqat sizniki: boshqa odamning
flag'i sizda ishlamaydi.

## 8. Nima bo'ldi?

Siz ikki yuzta javobdan atigi o'n ikkitasini o'zgartirdingiz. Modelning umumiy aniqligi deyarli o'zgarmadi,
ya'ni oddiy tekshiruvda hech kim hech narsani sezmaydi. Lekin siz tanlagan bitta nuqta bo'yicha model endi
xato javob beradi. Zaharlashning xavfi aynan shunda: zarar kichik va yashirin, lekin hujumchiga kerakli joyda.

Haqiqiy hayotda nishon biror aniq odamning arizasi, bitta aniq fayl yoki bitta aniq xabar bo'lishi mumkin.
Ma'lumot esa ko'pincha ochiq manbalardan yoki foydalanuvchilarning o'zidan yig'iladi, ya'ni unga yozish huquqini
qo'lga kiritish biz o'ylagandan osonroq.

Himoyachi nima qilishi mumkin? Avvalo, ma'lumot qayerdan kelganini va uni kim o'zgartira olishini nazorat
qilish kerak. Ikkinchidan, label'larni qo'shnilari bilan solishtirish mumkin: atrofidagi hamma nuqtalar bir
guruhda bo'lib, bitta-ikkitasi boshqa guruhda tursa, bu shubhali. E'tibor bering, siz hozir aynan shunday iz
qoldirdingiz. Uchinchidan, model qayta o'qitilganda faqat umumiy aniqlikka emas, alohida muhim misollardagi
javoblar o'zgarganiga ham qarash kerak.

## Qo'shimcha (xohlasangiz)

1. O'n ikkitadan kam label bilan uddalasa bo'ladimi? Eng kam nechta yetadi?
2. Teskarisini sinab ko'ring: nishonning yonidagi **boshqa** guruh nuqtalarini almashtiring. Nima bo'ladi va nega?
3. Tavakkal o'n ikkita label tanlab yuborib ko'ring. Aniqlik va nishon qanday o'zgardi?